
# GVH Diagonal Cubic 0.3.2.7.3.7.2.4 — Spatial-Gradient and Shift Reconstruction of Exact Canonical Constraint Densities

**Auteur :** Charlemagne O Laurince

`0.3.2.7.3.7.2.3` a laissé ouverts \(\mathcal V\), \(\mathcal C_i\) et \(R_{DD2}\).
Ici on restaure
\[
\mathcal D_\perp=\frac1N(\partial_t-\mathcal L_{\vec N})
\]
et on dérive exactement la densité de momentum à partir de la 1-forme canonique.

\[
\boxed{\mathrm{DISPERSION\_READY=False}}
\]


In [1]:

import sympy as sp, json
from pathlib import Path
print("GVH 0.3.2.7.3.7.2.4")
print("SymPy:", sp.__version__)


GVH 0.3.2.7.3.7.2.4
SymPy: 1.14.0



## 1. Vitesses normales ADM

Pour un scalaire \(s\),
\[
\mathcal D_\perp s=\frac{\dot s-N^iD_is}{N}.
\]

Pour un covecteur \(v_i\),
\[
\mathcal D_\perp v_i
=
\frac{\dot v_i-N^jD_jv_i-v_jD_iN^j}{N}.
\]

Et
\[
K_{ij}=\frac{\dot h_{ij}-\mathcal L_{\vec N}h_{ij}}{2N}.
\]


In [2]:

N,sdot,NdS = sp.symbols("N sdot NdS", nonzero=True)
Sperp=(sdot-NdS)/N
assert sp.simplify(N*Sperp-(sdot-NdS))==0
print("ADM normal derivative identity: PASS")


ADM normal derivative identity: PASS



## 2. Contribution scalaire à \(\mathcal C_i\)

\[
p_s\dot s=Np_s\mathcal D_\perp s+N^ip_sD_is
\]

donc
\[
\boxed{\mathcal C_i^{(s)}=p_sD_is}.
\]


In [3]:

ps,N1,N2,N3,Ds1,Ds2,Ds3 = sp.symbols("ps N1 N2 N3 Ds1 Ds2 Ds3")
expr=ps*(N1*Ds1+N2*Ds2+N3*Ds3)
candidate=N1*(ps*Ds1)+N2*(ps*Ds2)+N3*(ps*Ds3)
assert sp.expand(expr-candidate)==0
print("scalar momentum density: PASS")


scalar momentum density: PASS



## 3. Contribution du covecteur spatial

\[
(\mathcal L_{\vec N}v)_i=N^jD_jv_i+v_jD_iN^j.
\]

Après intégration par parties :
\[
\boxed{
\mathcal C_j^{(v)}
=
p_v^iD_jv_i-D_i(p_v^iv_j)
}.
\]


In [4]:

Nv=sp.symbols("N0:3"); pv=sp.symbols("p0:3"); vv=sp.symbols("v0:3")
Dv=[[sp.symbols(f"D{j}v{i}") for j in range(3)] for i in range(3)]
DN=[[sp.symbols(f"D{i}N{j}") for j in range(3)] for i in range(3)]
Dp=[sp.symbols(f"D{i}p{i}") for i in range(3)]

original=sum(pv[i]*(Nv[j]*Dv[i][j]+vv[j]*DN[j][i]) for i in range(3) for j in range(3))
Cvec=[]
for j in range(3):
    Cvec.append(sum(pv[i]*Dv[i][j]-(Dp[i]*vv[j]+pv[i]*Dv[j][i]) for i in range(3)))
bulk=sum(Nv[j]*Cvec[j] for j in range(3))
boundary=sum(DN[j][i]*pv[i]*vv[j]+Nv[j]*Dp[i]*vv[j]+Nv[j]*pv[i]*Dv[j][i]
             for i in range(3) for j in range(3))
assert sp.expand(original-bulk-boundary)==0
print("covector integration-by-parts identity: PASS")


covector integration-by-parts identity: PASS



## 4. Contribution métrique et densité totale

Le secteur métrique donne
\[
\mathcal C_i^{(h)}=-2D_j\pi^j{}_i.
\]

Ainsi
\[
\boxed{
\mathcal C_i
=
-2D_j\pi^j{}_i
+p_sD_is
+p_v^jD_iv_j
-D_j(p_v^jv_i)
}
\]
à des termes proportionnels aux contraintes primaires des multiplicateurs près.


In [5]:

Ci = {
 "metric":"-2 D_j pi^j_i",
 "scalar":"p_s D_i s",
 "covector":"p_v^j D_i v_j - D_j(p_v^j v_i)"
}
print(Ci)


{'metric': '-2 D_j pi^j_i', 'scalar': 'p_s D_i s', 'covector': 'p_v^j D_i v_j - D_j(p_v^j v_i)'}



## 5. Partie normale déjà acquise

De 7.2.3 :
\[
\boxed{
\mathcal C_{\perp,\rm kin}
=
\frac12P^TQ_{\rm total}^{-1}P
}.
\]

Le potentiel spatial EH apporte le terme en \({}^{(3)}R\) selon les conventions de normalisation.



## 6. Verrou du lapse : \(a_i^{(n)}=D_i\ln N\)

Les blocs complets hérités sont
\[
A=-\mathcal D_\perp s-v^ia_i^{(n)},
\]
\[
B_i=s\,a_i^{(n)}+\mathcal D_\perp v_i-K_i{}^jv_j,
\]
\[
C_i=-D_is-K_i{}^jv_j,
\]
\[
D_{ij}=D_iv_j+sK_{ij}.
\]

Or
\[
a_i^{(n)}=\frac{D_iN}{N}.
\]

Donc la contrainte normale complète exige la dérivée fonctionnelle
\[
\boxed{
\frac{\delta H}{\delta N}
=
\frac{\partial\mathcal H}{\partial N}
-
D_i\!\left(\frac{\partial\mathcal H}{\partial(D_iN)}\right)
+\cdots
}.
\]


In [6]:

x,a=sp.symbols("x a")
Nx=sp.Function("N")(x)
L=a*sp.diff(Nx,x)**2/Nx
alg=sp.diff(L,Nx)
EL=sp.simplify(alg-sp.diff(sp.diff(L,sp.diff(Nx,x)),x))
assert sp.simplify(EL-alg)!=0
print("lapse-gradient functional derivative test: PASS")


lapse-gradient functional derivative test: PASS



## 7. Statut des contraintes

\[
\boxed{\mathcal C_i:\ \text{EXPLICIT KINEMATIC FORM}}
\]

mais
\[
\boxed{
\mathcal C_\perp:
\text{PARTIAL — lapse-gradient functional variation OPEN}
}.
\]

Par conséquent
\[
\boxed{R_{DD2}:\ \text{OPEN / UNCOMPUTED}}.
\]


In [7]:

GATES={
 "Dperp_shift_reconstruction":True,
 "scalar_Ci_derived":True,
 "covector_Ci_derived":True,
 "metric_Ci_registered":True,
 "total_Ci_formula_explicit":True,
 "Cperp_kinetic_inherited":True,
 "lapse_gradient_obstruction_identified":True,
 "full_directional_spatial_potential":False,
 "full_functional_Cperp":False,
 "RDD2_computed":False,
 "hypersurface_algebra_closed":False
}
for k,v in GATES.items(): print(k,":",v)
FINAL_STATUS="PARTIAL-PASS-SHIFT-RECONSTRUCTION-CI-EXPLICIT_BLOCKED-LAPSE-GRADIENT-FUNCTIONAL-C-PERP"
DISPERSION_READY=False
assert DISPERSION_READY is False
print("FINAL STATUS:",FINAL_STATUS)
print("DISPERSION_READY =",DISPERSION_READY)


Dperp_shift_reconstruction : True
scalar_Ci_derived : True
covector_Ci_derived : True
metric_Ci_registered : True
total_Ci_formula_explicit : True
Cperp_kinetic_inherited : True
lapse_gradient_obstruction_identified : True
full_directional_spatial_potential : False
full_functional_Cperp : False
RDD2_computed : False
hypersurface_algebra_closed : False
FINAL STATUS: PARTIAL-PASS-SHIFT-RECONSTRUCTION-CI-EXPLICIT_BLOCKED-LAPSE-GRADIENT-FUNCTIONAL-C-PERP
DISPERSION_READY = False



## 8. Prochaine étape

### `0.3.2.7.3.7.2.5 — Full Lapse-Gradient Functional Variation and Normal Constraint Density`

Elle devra reconstruire le potentiel directionnel spatial complet, conserver
\[
a_i^{(n)}=D_iN/N,
\]
effectuer la variation fonctionnelle en \(N\), puis obtenir \(\mathcal C_\perp\) avant tout calcul des crochets hypersurface.


In [8]:

artifact={
 "notebook":"GVH_Diagonal_Cubic_0.3.2.7.3.7.2.4",
 "final_status":FINAL_STATUS,
 "Ci_formula":Ci,
 "Cperp_kinetic":"1/2 P^T Q_total^{-1} P",
 "Cperp_full_status":"OPEN_LAPSE_GRADIENT_FUNCTIONAL_VARIATION",
 "RDD2_status":"OPEN_UNCOMPUTED",
 "gates":GATES,
 "dispersion_ready":False,
 "next":"GVH_Diagonal_Cubic_0.3.2.7.3.7.2.5_Full_Lapse_Gradient_Functional_Variation_and_Normal_Constraint_Density.ipynb"
}
export_dir=Path.cwd()/ "gvh_exports"
export_dir.mkdir(exist_ok=True)
path=export_dir/"gvh_0.3.2.7.3.7.2.4_shift_spatial_constraint_density.json"
path.write_text(json.dumps(artifact,indent=2),encoding="utf-8")
print("Artifact:",path)


Artifact: /content/gvh_exports/gvh_0.3.2.7.3.7.2.4_shift_spatial_constraint_density.json



# Conclusion

La dépendance au shift est reconstruite canoniquement et
\[
\boxed{
\mathcal C_i
=
-2D_j\pi^j{}_i+p_sD_is+p_v^jD_iv_j-D_j(p_v^jv_i)
}
\]
est obtenue à des termes primaires près.

La densité normale complète reste bloquée par la variation fonctionnelle du lapse via
\[
a_i^{(n)}=D_iN/N.
\]

Verdict :
\[
\boxed{\text{PARTIAL PASS}},\qquad
\boxed{\mathrm{DISPERSION\_READY=False}}.
\]
